# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/yashalaf/flyrank-internship/blob/main/work/notebooks/w03_data_contract.ipynb)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Unit of analysis + time window

*One row = one what, over which dates? State it, then verify it below.*

**Unit of analysis, claimed: one row = one (report_date, client, content_item) combination**, from `fact_content_daily_performance`. Key columns are hashed (`client_hash_id`, `content_hash_id`), not plain `client_id`/`content_id` as I first assumed.

**Time window:** confirmed live, `fact_content_daily_performance_sample.parquet` (11,694,072 rows) covers **2026-06-01 to 2026-06-30**, exactly the "latest full month" the dataset card promises. Full history spans 19 monthly partitions, 2025-01 through 2026-03, plus this sample month, but it's an unbalanced panel, checked in Section 3/4.

**The grain claim above is not fully true.** Running the grain probe found real duplicates: `client_e00b29e582949543` has two rows for the same `(report_date, content_hash_id)` on 2026-06-13, across at least 5 different content items that day. That's not a hypothetical edge case, it actually happened, so "one row per day per client per content" is downgraded from a fact to a claim-with-an-exception until Section 3 figures out why.

In [1]:
# Setup: DuckDB reading directly from the gated Hugging Face warehouse.
# Token must come from getpass or Colab Secrets, never pasted in a cell (public repo).
import duckdb
from getpass import getpass
from huggingface_hub import HfApi

HF_TOKEN = getpass("HF read token: ")  # or: from google.colab import userdata; HF_TOKEN = userdata.get('HF_TOKEN')

# Don't guess the file path, list what's actually there first.
api = HfApi()
all_files = api.list_repo_files("FlyRank/internship-warehouse", repo_type="dataset", token=HF_TOKEN)
daily_files = [f for f in all_files if "fact_content_daily_performance" in f]
print(f"{len(daily_files)} files under fact_content_daily_performance:")
for f in sorted(daily_files)[:15]:
    print(" ", f)

sample_files = [f for f in daily_files if "sample" in f.lower()]
print("\nfiles matching 'sample':", sample_files)

con = duckdb.connect()
con.execute(f"CREATE SECRET (TYPE huggingface, TOKEN '{HF_TOKEN}')")
WAREHOUSE = "hf://datasets/FlyRank/internship-warehouse"

if sample_files:
    SAMPLE = f"{WAREHOUSE}/{sample_files[0]}"
else:
    SAMPLE = f"{WAREHOUSE}/fact_content_daily_performance/**/*.parquet"
print("\nusing path:", SAMPLE)

# Don't guess column names either, list the real schema before writing any query against it.
schema = con.sql(f"DESCRIBE SELECT * FROM read_parquet('{SAMPLE}') LIMIT 1").fetchdf()
print("\nreal columns:")
print(schema['column_name'].tolist())

# Claim check 1: row count of the sample table
row_count = con.sql(f"SELECT COUNT(*) AS n FROM read_parquet('{SAMPLE}')").fetchone()[0]
print("\nsample row count:", row_count)

# Claim check 2: date window actually covered by the sample
window = con.sql(f"SELECT MIN(report_date) AS min_date, MAX(report_date) AS max_date FROM read_parquet('{SAMPLE}')").fetchone()
print("sample date window:", window)

# Claim check 3: grain probe. Real key columns are client_hash_id / content_hash_id
# (hashed per the dataset card's "salted, namespaced, fingerprinted hash keys"), not client_id/content_id.
grain_violations = con.sql(f"""
    SELECT report_date, client_hash_id, content_hash_id, COUNT(*) c
    FROM read_parquet('{SAMPLE}')
    GROUP BY report_date, client_hash_id, content_hash_id
    HAVING c > 1
    LIMIT 5
""").fetchall()
print("\ngrain violations (should be empty):", grain_violations)


HF read token: ··········
19 files under fact_content_daily_performance:
  fact_content_daily_performance/month=2025-01/data_0.parquet
  fact_content_daily_performance/month=2025-02/data_0.parquet
  fact_content_daily_performance/month=2025-03/data_0.parquet
  fact_content_daily_performance/month=2025-04/data_0.parquet
  fact_content_daily_performance/month=2025-05/data_0.parquet
  fact_content_daily_performance/month=2025-06/data_0.parquet
  fact_content_daily_performance/month=2025-07/data_0.parquet
  fact_content_daily_performance/month=2025-08/data_0.parquet
  fact_content_daily_performance/month=2025-09/data_0.parquet
  fact_content_daily_performance/month=2025-10/data_0.parquet
  fact_content_daily_performance/month=2025-11/data_0.parquet
  fact_content_daily_performance/month=2025-12/data_0.parquet
  fact_content_daily_performance/month=2026-01/data_0.parquet
  fact_content_daily_performance/month=2026-02/data_0.parquet
  fact_content_daily_performance/month=2026-03/data_0.parqu

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

sample date window: (datetime.date(2026, 6, 1), datetime.date(2026, 6, 30))


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))


grain violations (should be empty): [(datetime.date(2026, 6, 13), 'client_1a730cb2640a1abf', 'content_1a0c3a8cc6bdd5bc', 2), (datetime.date(2026, 6, 13), 'client_1a730cb2640a1abf', 'content_8d5a5293688d7d8e', 2), (datetime.date(2026, 6, 13), 'client_1a730cb2640a1abf', 'content_b40e27e07768f907', 2), (datetime.date(2026, 6, 13), 'client_1a730cb2640a1abf', 'content_965b9031838c130f', 2), (datetime.date(2026, 6, 13), 'client_1a730cb2640a1abf', 'content_fa351551b6d49870', 2)]


## 2. Fields: feature / label / context / excluded

*Sort every field you plan to touch into these four buckets. Excluded needs a why.*

**Real schema, confirmed** (30 columns): `report_date`, `client_hash_id`, `content_hash_id`, `client_has_gsc`, `client_has_ga4`, `gsc_data_available`, `ga4_data_available`, `gsc_impressions`, `gsc_clicks`, `gsc_sum_position`, `gsc_avg_position`, `ga4_pageviews`, `ga4_sessions`, `ga4_users`, `ga4_engaged_sessions`, `ga4_total_engagement_sec`, `sessions_organic`, `sessions_direct`, `sessions_referral`, `sessions_social`, `sessions_paid`, `sessions_ai`, `ai_chatgpt`, `ai_perplexity`, `ai_gemini`, `ai_copilot`, `ai_claude`, `ai_meta`, `ai_other`, `scroll_events`, `month`.

- **Context** (grouping/joining/partitioning only, never a feature): `report_date`, `client_hash_id`, `content_hash_id`, `month` (partition column).
- **Availability flags** (their own bucket, not plain features): `client_has_gsc`, `client_has_ga4`, `gsc_data_available`, `ga4_data_available`. These say whether a row's other columns can be trusted, they gate which features are usable per row, they don't describe content performance themselves.
- **Feature** (knowable at prediction time, describes performance): `gsc_impressions`, `gsc_clicks`, `gsc_sum_position`, `gsc_avg_position`, all the `ga4_*` and `sessions_*` columns, the `ai_*` assistant-referral breakdown, `scroll_events`.
- **Label / proxy: there isn't one.** Unlike the starter CSV, which ships a ready-made `trend_direction`, this daily fact table has no pre-computed trend or decline column. Any label has to be derived by me, comparing rolling windows of these same feature columns across dates, not assumed to already exist. That's a real difference worth stating plainly rather than quietly importing the starter CSV's label concept where it doesn't apply.
- **Excluded**: any `ga4_*`/`sessions_*` column where `ga4_data_available` is false, and any `gsc_*` column where `gsc_data_available` is false, those aren't zero-filled, they're not measured. Also excluded: anything client-identifying beyond the hash.

In [2]:
# dim_clients cross-check: does client_has_gsc/client_has_ga4 in the fact table
# actually agree with dim_clients' access_profile?
client_files = [f for f in all_files if "dim_clients" in f]
print("dim_clients files:", client_files)
clients_df = con.sql(f"SELECT * FROM read_parquet('{WAREHOUSE}/{client_files[0]}')").fetchdf()
print(clients_df['access_profile'].value_counts())

check = con.sql(f"""
    SELECT client_has_gsc, client_has_ga4, gsc_data_available, ga4_data_available, COUNT(*) n
    FROM read_parquet('{SAMPLE}')
    GROUP BY 1, 2, 3, 4
    ORDER BY n DESC
""").fetchdf()
print("\navailability-flag combinations actually observed in the fact table:")
print(check)

dim_clients files: ['dim_clients.parquet']
access_profile
gsc_and_ga4                             53
no_search_or_analytics_access           26
gsc_only                                14
source_only_missing_client_dimension    10
ga4_only                                 1
Name: count, dtype: int64


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))


availability-flag combinations actually observed in the fact table:
   client_has_gsc  client_has_ga4  gsc_data_available  ga4_data_available  \
0            True            True               False               False   
1            True            True                True               False   
2            True           False               False                <NA>   
3            True           False                True                <NA>   
4            True            True                True                True   
5            True            True               False                True   

         n  
0  6283456  
1  2368462  
2  1441169  
3   956259  
4   554216  
5    90510  


## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

Every claim from sections 1 and 2 gets its query here, per the skill: *"A contract claim without a query next to it is a guess."*
First, chasing down the grain violation from Section 1, since an unexplained "sometimes there are 2 rows" is worse than either "always 1" or a documented reason for 2.

In [3]:
# Diagnose the duplicate rows found in Section 1: what actually differs between them?
dupe_rows = con.sql(f"""
    SELECT *
    FROM read_parquet('{SAMPLE}')
    WHERE client_hash_id = 'client_e00b29e582949543'
      AND content_hash_id = 'content_e3491394a9f3e2b3'
      AND report_date = DATE '2026-06-13'
""").fetchdf()
print(dupe_rows.T)

# How widespread is this? Total violating groups and their share of the sample.
n_violations = con.sql(f"""
    SELECT COUNT(*) FROM (
        SELECT report_date, client_hash_id, content_hash_id
        FROM read_parquet('{SAMPLE}')
        GROUP BY 1, 2, 3
        HAVING COUNT(*) > 1
    )
""").fetchone()[0]
print(f"\ntotal (date, client, content) groups with more than 1 row: {n_violations}")

cols = schema['column_name'].tolist()

# Missingness per column.
missing_overall = con.sql(f"""
    SELECT {", ".join([f"AVG(CASE WHEN {c} IS NULL THEN 1.0 ELSE 0 END) AS {c}_missing_rate" for c in cols])}
    FROM read_parquet('{SAMPLE}')
""").fetchdf()
print("\noverall missingness by column:")
print(missing_overall.T)

# Rows per client.
per_client_rows = con.sql(f"""
    SELECT client_hash_id, COUNT(*) n_rows
    FROM read_parquet('{SAMPLE}')
    GROUP BY client_hash_id
    ORDER BY n_rows DESC
    LIMIT 10
""").fetchdf()
print("\nrows per client (top 10):")
print(per_client_rows)

# Window check per client.
window_per_client = con.sql(f"""
    SELECT client_hash_id, MIN(report_date) AS min_date, MAX(report_date) AS max_date, COUNT(*) n
    FROM read_parquet('{SAMPLE}')
    GROUP BY client_hash_id
    ORDER BY min_date
    LIMIT 10
""").fetchdf()
print("\nper-client date windows (earliest 10):")
print(window_per_client)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

                                                 0                         1
report_date                    2026-06-13 00:00:00       2026-06-13 00:00:00
client_hash_id             client_e00b29e582949543   client_e00b29e582949543
content_hash_id           content_e3491394a9f3e2b3  content_e3491394a9f3e2b3
client_has_gsc                                True                      True
client_has_ga4                                True                      True
gsc_data_available                           False                     False
ga4_data_available                           False                     False
gsc_impressions                                  0                         0
gsc_clicks                                       0                         0
gsc_sum_position                                 0                         0
gsc_avg_position                               NaN                       NaN
ga4_pageviews                                    0                         0

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))


total (date, client, content) groups with more than 1 row: 6390


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))


overall missingness by column:
                                                  0
report_date_missing_rate               0.000000e+00
client_hash_id_missing_rate            0.000000e+00
content_hash_id_missing_rate           0.000000e+00
client_has_gsc_missing_rate            0.000000e+00
client_has_ga4_missing_rate            0.000000e+00
gsc_data_available_missing_rate        0.000000e+00
ga4_data_available_missing_rate        2.050122e-01
gsc_impressions_missing_rate           0.000000e+00
gsc_clicks_missing_rate                0.000000e+00
gsc_sum_position_missing_rate          3.420536e-07
gsc_avg_position_missing_rate          6.683019e-01
ga4_pageviews_missing_rate             2.050122e-01
ga4_sessions_missing_rate              2.050122e-01
ga4_users_missing_rate                 2.050122e-01
ga4_engaged_sessions_missing_rate      2.050122e-01
ga4_total_engagement_sec_missing_rate  2.050122e-01
sessions_organic_missing_rate          2.050122e-01
sessions_direct_missing_rate    

## 4. Data limits

*What can this data never tell you? Unbalanced history, GSC-only early rows, window overlaps.*

**What this data can never tell me, confirmed against real output:**

- **True duplicate rows exist, and they're identical, not a hidden dimension.** The 2026-06-13 duplicate for client_e00b29e582949543 / content_e3491394a9f3e2b3 matches on every single column, including gsc_data_available=False and all-zero metrics. This is the same row twice, not a second legitimate entry, pointing to a pipeline defect (double-write/double-ingest) rather than a real second measurement. Scope: 6,390 (date, client, content) groups have more than one row, out of 11.69M rows total, about 0.055% of the sample, small, but any SUM-based aggregation without a DISTINCT or GROUP BY dedup step first will silently double-count exactly these rows.

- **ga4_data_available has three real states, not two, and they mean different things.** True (644,726 rows) is real measured traffic, avg 4.28 sessions. False (8,651,918 rows) is a genuine, trustworthy zero, ga4_sessions is exactly 0 for all of them, not null, meaning GA4 was checked and there really was no traffic. NULL (2,397,428 rows, 20.5%) is the only state that means "not measured", ga4_sessions and every other GA4 column is null for 100% of these rows. My first draft of this section wrongly treated False and NULL as the same thing ("unavailable"); they're not, False is a real observation, NULL is a missing one, and collapsing them would throw away 8.65M rows of legitimate zero-traffic signal.

- **gsc_avg_position's missingness is fully explained by zero impressions, confirmed exactly.** Of the 7,815,135 rows with zero gsc_impressions, all 7,815,135 have a null gsc_avg_position (mathematically, there's no position to average when there were no impressions). Of the 3,878,937 rows with nonzero impressions, only 35 have a null position, a tiny residual edge case, not a pattern. This also revealed that gsc_data_available lines up exactly with zero vs. nonzero impressions across the whole 11.69M-row sample (7,815,135 + 3,878,937), so gsc_data_available isn't an independent measurement flag here, it's mechanically tied to whether any impressions happened that day.

- **The panel is unbalanced even within one month.** Per-client row counts range from under a thousand to 965,925 in this single sample month. Most clients span the full 2026-06-01 to 2026-06-30 window, but at least one (client_cd12bcfd98942aa1) stops at 2026-06-27, three days short. Any per-client comparison needs to check both row count and actual date range.

- **No ready-made label, still true.** There's no trend/decline column at this grain. A label has to be built from these columns across a rolling window myself, and that derivation is part of the contract, not borrowed from the starter CSV's trend_direction.

- **Window overlap with fact_content_query_90d remains an untested, live risk**, not touched by this run, still worth flagging before combining tables for any labeling work.

In [4]:
# Confirm: does ga4_data_available behave as NULL (unmeasured) vs False (measured, no data)?
avail_check = con.sql(f"""
    SELECT
        ga4_data_available,
        COUNT(*) n_rows,
        SUM(CASE WHEN ga4_sessions IS NULL THEN 1 ELSE 0 END) AS null_ga4_sessions,
        AVG(ga4_sessions) AS avg_ga4_sessions
    FROM read_parquet('{SAMPLE}')
    GROUP BY ga4_data_available
""").fetchdf()
print(avail_check)

# Confirm: gsc_avg_position nulls line up with zero gsc_impressions, not with gsc_data_available.
pos_check = con.sql(f"""
    SELECT
        gsc_data_available,
        CASE WHEN gsc_impressions = 0 THEN 'zero_impressions' ELSE 'has_impressions' END AS impressions_bucket,
        COUNT(*) n_rows,
        SUM(CASE WHEN gsc_avg_position IS NULL THEN 1 ELSE 0 END) AS null_avg_position
    FROM read_parquet('{SAMPLE}')
    GROUP BY 1, 2
""").fetchdf()
print(pos_check)

   ga4_data_available   n_rows  null_ga4_sessions  avg_ga4_sessions
0               False  8651918                0.0          0.000000
1                <NA>  2397428          2397428.0               NaN
2                True   644726                0.0          4.280521
   gsc_data_available impressions_bucket   n_rows  null_avg_position
0               False   zero_impressions  7815135          7815135.0
1                True    has_impressions  3878937               35.0


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.